<a href="https://colab.research.google.com/github/inoue0426/llm-tuning-playground/blob/main/notebooks/11_ctd_multiseed_split_runner_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 11 - CTD multiseed and split robustness runner

Runs a compact reproducibility study for Vanilla vs Robust SFT. The notebook is self-contained. It supports chemical-, gene-, and disease-disjoint evaluation and saves raw and summary CSV files.

Default pilot: 3 seeds x chemical-disjoint. Set RUN_ALL_SPLITS = True to add gene- and disease-disjoint runs.

In [1]:
!pip -q install -U "transformers>=4.55,<5" "datasets>=3.6,<5" "peft>=0.17,<1" "trl==0.29.1" "accelerate>=1.10,<2" "bitsandbytes>=0.46,<1" "torchao>=0.16,<1"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 150.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 115.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 46.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.24.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import os
import random
import re
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from google.colab import files

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab.")

CHEM_GENE = "/content/CTD_chem_gene_ixns.tsv.gz"
GENE_DISEASE = "/content/CTD_curated_genes_diseases.tsv.gz"
if not os.path.exists(CHEM_GENE) or not os.path.exists(GENE_DISEASE):
    print("Upload CTD_chem_gene_ixns.tsv.gz and CTD_curated_genes_diseases.tsv.gz")
    files.upload()
assert os.path.exists(CHEM_GENE) and os.path.exists(GENE_DISEASE)
print("GPU:", torch.cuda.get_device_name(0))


Upload CTD_chem_gene_ixns.tsv.gz and CTD_curated_genes_diseases.tsv.gz


Saving CTD_curated_genes_diseases.tsv.gz to CTD_curated_genes_diseases.tsv.gz
Saving CTD_chem_gene_ixns.tsv.gz to CTD_chem_gene_ixns.tsv.gz
GPU: NVIDIA L4


In [3]:
chem_cols = ["ChemicalName","ChemicalID","CasRN","GeneSymbol","GeneID","GeneForms","Organism","OrganismID","Interaction","InteractionActions","PubMedIDs"]
gd_cols = ["GeneSymbol","GeneID","DiseaseName","DiseaseID","DirectEvidence","InferenceChemicalName","InferenceChemicalID","OmimIDs","PubMedIDs"]
chem = pd.read_csv(CHEM_GENE, sep="\t", comment="#", header=None, names=chem_cols, dtype=str, low_memory=False)
gd = pd.read_csv(GENE_DISEASE, sep="\t", comment="#", header=None, names=gd_cols, dtype=str, low_memory=False)
chem = chem[chem["OrganismID"].fillna("").str.strip().eq("9606")].copy()
chem = chem.dropna(subset=["ChemicalName","ChemicalID","GeneSymbol","GeneID"]).copy()
gd = gd.dropna(subset=["GeneID","DiseaseName","DiseaseID"]).copy()
chem["GeneID"] = chem["GeneID"].str.replace(r"\.0$", "", regex=True)
gd["GeneID"] = gd["GeneID"].str.replace(r"\.0$", "", regex=True)
gd = gd.drop_duplicates(["GeneID","DiseaseID"])
pairs = chem.merge(gd[["GeneID","DiseaseName","DiseaseID"]], on="GeneID", how="inner")
pairs = pairs[["ChemicalName","ChemicalID","GeneSymbol","GeneID","DiseaseName","DiseaseID"]].dropna()
pairs = pairs.drop_duplicates(["ChemicalID","GeneID","DiseaseID"]).reset_index(drop=True)
print("2-hop paths:", len(pairs))


2-hop paths: 4861330


In [4]:
SEEDS = [1, 2, 3]
RUN_ALL_SPLITS = False
MAX_TRAIN = 1200
MAX_EVAL = 100
MAX_STEPS = 60
DISTRACTOR_K = 5
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftModel
from trl import SFTConfig, SFTTrainer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
def render(prompt, answer=None):
    messages = [{"role": "user", "content": prompt}]
    if answer is not None:
        messages.append({"role": "assistant", "content": answer})
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=(answer is None))

def norm(x):
    return re.sub(r"[^a-z0-9]+", " ", str(x).lower()).strip()

def clean_prompt(row):
    return (f"Evidence 1: {row.ChemicalName} has a chemical-gene relationship with {row.GeneSymbol}.\n"
            f"Evidence 2: {row.GeneSymbol} is linked to disease {row.DiseaseName}.\n"
            f"Question: What disease is supported for {row.ChemicalName} through {row.GeneSymbol}? Return Disease: <name> and Path: Chemical -> Gene -> Disease.")

def clean_answer(row):
    return f"Disease: {row.DiseaseName}. Path: {row.ChemicalName} -> {row.GeneSymbol} -> {row.DiseaseName}."


In [6]:
def build_candidates(df):
    return list({(str(g), str(d)) for g, d in df[["GeneSymbol","DiseaseName"]].itertuples(index=False, name=None)})

def make_distractor(row, candidates, k, rng):
    picked=[]; seen=set(); tries=0
    while len(picked)<k and tries<1000:
        g,d=rng.choice(candidates); tries+=1
        if g==row.GeneSymbol or d==row.DiseaseName or (g,d) in seen: continue
        seen.add((g,d)); picked.append((g,d))
    if len(picked)<k: return None
    edges=[f"{row.GeneSymbol} -> {row.DiseaseName}"]+[f"{g} -> {d}" for g,d in picked]; rng.shuffle(edges)
    return (f"Evidence A: {row.ChemicalName} has a chemical-gene relationship with {row.GeneSymbol}.\n"
            +"Gene-disease evidence:\n- "+"\n- ".join(edges)+"\n"
            +f"Question: What disease is supported for {row.ChemicalName} through {row.GeneSymbol}? Return Disease: <name> and Path: Chemical -> Gene -> Disease.")

def make_no_path(row, candidates, k, rng):
    picked=[]; seen=set(); tries=0
    while len(picked)<k and tries<1000:
        g,d=rng.choice(candidates); tries+=1
        if g==row.GeneSymbol or (g,d) in seen: continue
        seen.add((g,d)); picked.append((g,d))
    if len(picked)<k: return None
    edges=[f"{g} -> {d}" for g,d in picked]; rng.shuffle(edges)
    return (f"Evidence A: {row.ChemicalName} has a chemical-gene relationship with {row.GeneSymbol}.\n"
            +"Gene-disease evidence:\n- "+"\n- ".join(edges)+"\n"
            +f"Question: What disease is supported for {row.ChemicalName} through {row.GeneSymbol}? If the evidence does not support a path, return No supported path.")


In [7]:
def split_paths(df, unit, seed):
    rng=random.Random(seed); vals=df[unit].drop_duplicates().tolist(); rng.shuffle(vals)
    n_eval=max(1,int(0.2*len(vals))); eval_units=set(vals[:n_eval])
    train=df[~df[unit].isin(eval_units)].sample(frac=1.0,random_state=seed).head(MAX_TRAIN).reset_index(drop=True)
    ev=df[df[unit].isin(eval_units)].sample(frac=1.0,random_state=seed+100).head(MAX_EVAL).reset_index(drop=True)
    return train,ev

def make_dataset(train_df,candidates,seed,robust):
    rng=random.Random(seed); rows=[]
    for row in train_df.itertuples(index=False):
        if robust and rng.random()<0.5:
            p=make_distractor(row,candidates,DISTRACTOR_K,rng) or clean_prompt(row)
            a=clean_answer(row)
        else:
            p,a=clean_prompt(row),clean_answer(row)
        rows.append({"text":render(p,a)})
    return Dataset.from_list(rows)


In [8]:
def generate_batched(model,prompts,batch_size=16,max_new_tokens=64):
    out_text=[]; model.eval()
    for s in range(0,len(prompts),batch_size):
        enc=tokenizer([render(p) for p in prompts[s:s+batch_size]],return_tensors="pt",padding=True,truncation=True,max_length=384)
        enc={k:v.to(model.device) for k,v in enc.items()}
        with torch.inference_mode():
            out=model.generate(**enc,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tokenizer.pad_token_id)
        n=enc["input_ids"].shape[1]
        out_text.extend(tokenizer.batch_decode(out[:,n:],skip_special_tokens=True))
    return out_text

def train_one(ds,outdir):
    model=AutoModelForCausalLM.from_pretrained(MODEL_NAME,dtype=dtype).cuda(); model.config.use_cache=False
    lora=LoraConfig(r=8,lora_alpha=16,lora_dropout=0.05,target_modules=["q_proj","k_proj","v_proj","o_proj"],bias="none",task_type="CAUSAL_LM")
    args=SFTConfig(output_dir=outdir,per_device_train_batch_size=4,gradient_accumulation_steps=2,max_steps=MAX_STEPS,learning_rate=2e-4,logging_steps=20,save_strategy="no",report_to="none",packing=False,gradient_checkpointing=False,fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported())
    trainer=SFTTrainer(model=model,args=args,train_dataset=ds,processing_class=tokenizer,peft_config=lora)
    trainer.train(); trainer.save_model(outdir); tokenizer.save_pretrained(outdir)
    del trainer,model; torch.cuda.empty_cache()

def score_positive(items,preds):
    return float(np.mean([norm(i["target_disease"]) in norm(p) for i,p in zip(items,preds)])) if items else float("nan")

def score_no_path(items,preds):
    return float(np.mean(["no supported path" in norm(p) for p in preds])) if items else float("nan")


In [9]:
def run_split(split_unit,seed):
    train_df,eval_df=split_paths(pairs,split_unit,seed); candidates=build_candidates(pairs); rng=random.Random(seed+999)
    pos=[]; dist=[]; nopath=[]
    for row in eval_df.itertuples(index=False):
        base={"target_disease":row.DiseaseName}
        pos.append({**base,"prompt":clean_prompt(row)})
        dp=make_distractor(row,candidates,DISTRACTOR_K,rng)
        if dp: dist.append({**base,"prompt":dp})
        nprompt=make_no_path(row,candidates,DISTRACTOR_K,rng)
        if nprompt: nopath.append({**base,"prompt":nprompt})
    records=[]
    for condition,robust in [("vanilla",False),("robust",True)]:
        ds=make_dataset(train_df,candidates,seed+(1000 if robust else 0),robust)
        outdir=f"./outputs/11_{split_unit}_{seed}_{condition}"
        train_one(ds,outdir)
        base=AutoModelForCausalLM.from_pretrained(MODEL_NAME,dtype=dtype).cuda(); model=PeftModel.from_pretrained(base,outdir).eval()
        for metric,items,scorer in [("clean",pos,score_positive),(f"distractor_{DISTRACTOR_K}",dist,score_positive),(f"no_path_{DISTRACTOR_K}",nopath,score_no_path)]:
            preds=generate_batched(model,[x["prompt"] for x in items]); score=scorer(items,preds)
            records.append({"split":split_unit,"seed":seed,"condition":condition,"metric":metric,"score":score,"n_eval":len(items)})
        del model,base; torch.cuda.empty_cache()
    return records


In [ ]:
runs=["ChemicalID"] if not RUN_ALL_SPLITS else ["ChemicalID","GeneID","DiseaseID"]
records=[]
for split_unit in runs:
    for seed in SEEDS:
        print(f"\n=== split={split_unit}, seed={seed} ===")
        records.extend(run_split(split_unit,seed))

results_df=pd.DataFrame(records)
results_df.to_csv("11_results.csv",index=False)
summary_df=(results_df.groupby(["split","condition","metric"],as_index=False)["score"].agg(mean="mean",std="std",n="count"))
summary_df.to_csv("11_summary.csv",index=False)
print(summary_df.to_string(index=False))
print("Saved: 11_results.csv and 11_summary.csv")


In [13]:
pd.read_csv("11_results.csv")

,split,seed,condition,metric,score,n_eval
0,ChemicalID,1,vanilla,clean,0.99,100
1,ChemicalID,1,vanilla,distractor_5,0.70,100
2,ChemicalID,1,vanilla,no_path_5,0.00,100
3,ChemicalID,1,robust,clean,0.97,100
4,ChemicalID,1,robust,distractor_5,0.96,100
5,ChemicalID,1,robust,no_path_5,0.00,100
6,ChemicalID,2,vanilla,clean,1.00,100
7,ChemicalID,2,vanilla,distractor_5,0.71,100
8,ChemicalID,2,vanilla,no_path_5,0.00,100
9,ChemicalID,2,robust,clean,0.99,100
